# Klasterovanje u bioinformatici

Algoritmi:
1. **FarthestFirstTraversal** — heuristika za k-centar problem
2. **Lloyd algoritam (K-means)** — iterativno klasterovanje sa centrom gravitacije
3. **Meki K-means (Soft K-means)** — EM pristup sa matricom odgovornosti
4. **Hijerarhijsko klasterovanje** — aglomerativno, D_min i D_avg (UPGMA)
5. **CAST** — klasterovanje zasnovano na grafu sličnosti (pokvarene klike)
6. **DBSCAN** — density-based, sa identifikacijom šuma
7. **DIANA** — divisive hijerarhijsko (top-down)
8. **Louvain / Leiden** — community detection, optimizacija modularnosti
9. **Brute-force K-means** — iscrpno traženje optimuma (referenca za Lloyd)

In [1]:
import math
import random

## Pomoćne funkcije za rad

In [2]:
'''Euklidsko rastojanje izmedu dve tacke'''
def euclidean_distance(a, b):
    return math.sqrt(sum((ai - bi) ** 2 for ai, bi in zip(a, b)))


'''Centar gravitacije skupa tacaka'''
def centroid(points):
    m = len(points[0])
    n = len(points)
    return [sum(t[d] for t in points) / n for d in range(m)]


'''Pearsonov koeficijent korelacije izmedju dva vektora'''
def pearson_correlation(x, y):
    n = len(x)
    if n == 0:
        return 0.0
    
    mean_x = sum(x) / n
    mean_y = sum(y) / n
    
    numerator = sum((x[i] - mean_x) * (y[i] - mean_y) for i in range(n))
    denom_x = math.sqrt(sum((x[i] - mean_x) ** 2 for i in range(n)))
    denom_y = math.sqrt(sum((y[i] - mean_y) ** 2 for i in range(n)))
    
    if denom_x == 0 or denom_y == 0:
        return 0.0
    
    return numerator / (denom_x * denom_y)

## 1. FarthestFirstTraversal

Heuristika za **k-centar problem klasterovanja** — bira centre iz podataka tako da minimizuje maksimalno rastojanje bilo koje tačke do njenog najbližeg centra.

**Algoritam:**
1. Izaberi proizvoljnu tačku kao prvi centar
2. Dok nemamo k centara:
   - Dodaj tačku koja je **najudaljenija** od svih dosadašnjih centara

In [3]:
class FarthestFirstTraversal:
    def __init__(self, k):
        self.k = k
        self.centers = []
    
    '''Rastojanje tacke do najblizeg centra'''
    def distance_to_centers(self, point):
        return min(euclidean_distance(point, c) for c in self.centers)
    
    '''Pokretanje algoritma'''
    def fit(self, data):
        # Izaberi prvu tacku proizvoljno
        self.centers = [data[0][:]]
        
        while len(self.centers) < self.k:
            # Pronadi tacku najudaljeniju od svih centara
            max_dist = -1
            farthest = None
            
            for point in data:
                r = self.distance_to_centers(point)
                if r > max_dist:
                    max_dist = r
                    farthest = point
            
            self.centers.append(farthest[:])
        
        return self.centers
    
    '''Dodela tacaka klasterima (po najblizem centru)'''
    def assign_clusters(self, data):
        labels = []
        for point in data:
            min_r = float('inf')
            closest = 0
            for i, c in enumerate(self.centers):
                r = euclidean_distance(point, c)
                if r < min_r:
                    min_r = r
                    closest = i
            labels.append(closest)
        return labels

## 2. Lloyd algoritam (K-means)

Najpoznatija heuristika za **k-means problem klasterovanja** — minimizuje kvadratnu grešku distorzije.

**Algoritam** iterativno ponavlja dva koraka:
1. **Centri → Klasteri:** Dodeli svaku tačku njenom najbližem centru
2. **Klasteri → Centri:** Ažuriraj svaki centar na centar gravitacije svog klastera

Algoritam konvergira kada se centri prestanu menjati.

**k-means++ inicijalizacija:** Umesto nasumičnog biranja, svaka sledeća tačka se bira sa verovatnoćom proporcionalnom d(tačka, centri)².

In [4]:
class KMeans:
    def __init__(self, k, max_iter=100, tolerance=1e-6):
        self.k = k
        self.max_iter = max_iter
        self.tolerance = tolerance
        self.centers = []
        self.labels = []
    
    '''k-means++ inicijalizacija centara'''
    def init_centers(self, data):
        self.centers = [data[random.randint(0, len(data) - 1)][:]]
        
        for _ in range(1, self.k):
            # Kvadrat rastojanja svake tacke do najblizeg centra
            distances = []
            for point in data:
                min_r = min(euclidean_distance(point, c) for c in self.centers)
                distances.append(min_r ** 2)
            
            # Biraj sledeci centar sa verovatnocom ~ d^2
            total = sum(distances)
            r = random.random() * total
            cumulative = 0
            for i, d in enumerate(distances):
                cumulative += d
                if cumulative >= r:
                    self.centers.append(data[i][:])
                    break
    
    '''Korak "Centri -> Klasteri": dodeli svaku tacku najblizem centru'''
    def assign_clusters(self, data):
        self.labels = []
        for point in data:
            min_r = float('inf')
            closest = 0
            for i, c in enumerate(self.centers):
                r = euclidean_distance(point, c)
                if r < min_r:
                    min_r = r
                    closest = i
            self.labels.append(closest)
    
    '''Korak "Klasteri -> Centri": azuriraj centre kao centre gravitacije'''
    def update_centers(self, data):
        new_centers = []
        for i in range(self.k):
            members = [data[j] for j in range(len(data)) if self.labels[j] == i]
            if members:
                new_centers.append(centroid(members))
            else:
                new_centers.append(self.centers[i][:])
        return new_centers
    
    '''Kvadratna greska distorzije'''
    def distortion(self, data):
        total = 0
        for j, point in enumerate(data):
            r = euclidean_distance(point, self.centers[self.labels[j]])
            total += r ** 2
        return total / len(data)
    
    '''Pokretanje Lloyd algoritma'''
    def fit(self, data):
        self.init_centers(data)
        
        for iteration in range(self.max_iter):
            # Centri → Klasteri
            self.assign_clusters(data)
            
            # Klasteri → Centri
            new_centers = self.update_centers(data)
            
            # Provera konvergencije
            shift = max(euclidean_distance(self.centers[i], new_centers[i]) 
                         for i in range(self.k))
            self.centers = new_centers
            
            if shift < self.tolerance:
                return iteration + 1
        
        return self.max_iter

## 3. Meki K-means (Expectation Maximization)

Za razliku od Lloyd algoritma koji **kruto** dodeljuje svaku tačku jednom klasteru, meki K-means dodeljuje **odgovornosti** (responsibilities) — koliko svaki centar "privlači" svaku tačku.

**Partition funkcija (Bolcmanova raspodela):**

$$HiddenMatrix_{i,j} = \frac{e^{-\beta \cdot d(Data_j, x_i)}}{\sum_{c} e^{-\beta \cdot d(Data_j, x_c)}}$$

- **E-korak:** Izračunaj matricu odgovornosti iz trenutnih centara
- **M-korak:** Izračunaj nove centre kao ponderisane centre gravitacije

Parametar **β (beta)** kontroliše krutost: veći β → krutije dodele (bliže Lloyd-u).

In [5]:
class SoftKMeans:
    def __init__(self, k, beta=2.0, max_iter=100, tolerance=1e-4):
        self.k = k
        self.beta = beta
        self.max_iter = max_iter
        self.tolerance = tolerance
        self.centers = []
        self.responsibility_matrix = []   # k × n matrica (HiddenMatrix)
    
    '''Inicijalizacija centara (k-means++)'''
    def init_centers(self, data):
        self.centers = [data[random.randint(0, len(data) - 1)][:]]
        
        for _ in range(1, self.k):
            distances = []
            for point in data:
                min_r = min(euclidean_distance(point, c) for c in self.centers)
                distances.append(min_r ** 2)
            
            total = sum(distances)
            r = random.random() * total
            cumulative = 0
            for i, d in enumerate(distances):
                cumulative += d
                if cumulative >= r:
                    self.centers.append(data[i][:])
                    break
    
    '''E-korak: izracunaj matricu odgovornosti (HiddenMatrix)'''
    def e_step(self, data):
        n = len(data)
        self.responsibility_matrix = [[0.0] * n for _ in range(self.k)]
        
        for j in range(n):
            # Izracunaj eksponente za svaki centar
            exponents = []
            for i in range(self.k):
                d = euclidean_distance(data[j], self.centers[i])
                exponents.append(-self.beta * d)
            
            # Numericka stabilnost: oduzmi maksimum pre exp()
            max_val = max(exponents)
            exp_vals = [math.exp(e - max_val) for e in exponents]
            sum_total = sum(exp_vals)
            
            for i in range(self.k):
                self.responsibility_matrix[i][j] = exp_vals[i] / sum_total
    
    '''M-korak: azuriraj centre kao ponderisane centre gravitacije'''
    def m_step(self, data):
        n = len(data)
        m = len(data[0])
        new_centers = []
        
        for i in range(self.k):
            sum_weights = sum(self.responsibility_matrix[i][j] for j in range(n))
            
            if sum_weights == 0:
                new_centers.append(self.centers[i][:])
                continue
            
            new_center = []
            for dim in range(m):
                weighted_sum = sum(
                    self.responsibility_matrix[i][j] * data[j][dim] for j in range(n)
                )
                new_center.append(weighted_sum / sum_weights)
            new_centers.append(new_center)
        
        return new_centers
    
    '''Pokretanje EM algoritma'''
    def fit(self, data):
        self.init_centers(data)
        
        for iteration in range(self.max_iter):
            self.e_step(data)
            
            old_centers = [c[:] for c in self.centers]
            self.centers = self.m_step(data)
            
            # Provera konvergencije
            shift = max(euclidean_distance(old_centers[i], self.centers[i]) 
                         for i in range(self.k))
            
            if shift < self.tolerance:
                return iteration + 1
        
        return self.max_iter
    
    '''Krute labele (centar sa najvecom odgovornoscu)'''
    def hard_labels(self):
        n = len(self.responsibility_matrix[0])
        labels = []
        for j in range(n):
            max_resp = -1
            best = 0
            for i in range(self.k):
                if self.responsibility_matrix[i][j] > max_resp:
                    max_resp = self.responsibility_matrix[i][j]
                    best = i
            labels.append(best)
        return labels
    
    '''Odgovornosti svih centara za datu tacku'''
    def responsibilities(self, point_idx):
        return [self.responsibility_matrix[i][point_idx] for i in range(self.k)]

## 4. Hijerarhijsko klasterovanje

Umesto fiksnog broja klastera, gradi **stablo (dendrogram)** od dna nagore spajanjem najbližih klastera.

**Algoritam:**
1. Kreni sa n jednoelementnih klastera
2. Pronađi dva najbliža klastera i spoj ih
3. Ponavljaj dok ne ostane jedan klaster

**Funkcije rastojanja između klastera:**
- $D_{min}(C_1, C_2)$ = minimalno rastojanje između bilo kog para tačaka
- $D_{avg}(C_1, C_2)$ = prosečno rastojanje svih parova (UPGMA)

In [6]:
'''Pomocna klasa za reprezentaciju klastera u hijerarhijskom klasterovanju'''
class HCluster:
    def __init__(self, elements, height=0):
        self.elements = elements
        self.height = height
        self.left = None
        self.right = None
    
    def __str__(self):
        return f'{self.elements}'
    
    '''D_min: minimalno rastojanje izmedju dva klastera'''
    @staticmethod
    def d_min(c1, c2, D):
        min_r = float('inf')
        for i in c1.elements:
            for j in c2.elements:
                if D[i][j] < min_r:
                    min_r = D[i][j]
        return min_r
    
    '''D_avg (UPGMA): prosecno rastojanje izmedju dva klastera'''
    @staticmethod
    def d_avg(c1, c2, D):
        total = 0
        for i in c1.elements:
            for j in c2.elements:
                total += D[i][j]
        return total / (len(c1.elements) * len(c2.elements))


class HierarchicalClustering:
    def __init__(self, method='avg'):
        self.method = method    # 'min' ili 'avg'
        self.root = None
        self.history = []      # redosled spajanja
    
    '''Pronalazenje dva najbliza klastera'''
    def two_closest(self, clusters, D):
        min_r = float('inf')
        min_i, min_j = 0, 1
        
        for i in range(len(clusters)):
            for j in range(i + 1, len(clusters)):
                if self.method == 'min':
                    r = HCluster.d_min(clusters[i], clusters[j], D)
                else:
                    r = HCluster.d_avg(clusters[i], clusters[j], D)
                
                if r < min_r:
                    min_r = r
                    min_i, min_j = i, j
        
        return min_i, min_j, min_r
    
    '''Pokretanje hijerarhijskog klasterovanja'''
    def fit(self, D):
        n = len(D)
        clusters = [HCluster([i], 0) for i in range(n)]
        self.history = []
        
        while len(clusters) > 1:
            i, j, distance = self.two_closest(clusters, D)
            
            # Spoj dva klastera
            ci = clusters[i]
            cj = clusters[j]
            novi = HCluster(ci.elements + cj.elements, distance / 2)
            novi.left = ci
            novi.right = cj
            
            self.history.append((ci.elements, cj.elements, distance))
            
            # Ukloni stare, dodaj novi
            clusters = [clusters[x] for x in range(len(clusters)) if x != i and x != j]
            clusters.append(novi)
        
        self.root = clusters[0]
        return self.root
    
    '''Izdvajanje k klastera iz dendrograma (seci stablo)'''
    def extract_clusters(self, k):
        # BFS niz cvorova, seci kad dodes do k
        nodes = [self.root]
        
        while len(nodes) < k:
            # Pronadi cvor sa najvecom visinom (za deljenje)
            max_h = -1
            max_idx = 0
            for idx, c in enumerate(nodes):
                if c.left is not None and c.height > max_h:
                    max_h = c.height
                    max_idx = idx
            
            node = nodes[max_idx]
            if node.left is None:
                break
            
            nodes.pop(max_idx)
            nodes.append(node.left)
            nodes.append(node.right)
        
        return [c.elements for c in nodes]
    
    '''Ispis dendrograma'''
    def print_dendrogram(self, names=None):
        for ci, cj, dist in self.history:
            if names:
                name_i = [names[x] for x in ci]
                name_j = [names[x] for x in cj]
                print(f'  Spoj {name_i} + {name_j}  (rastojanje: {dist:.3f})')
            else:
                print(f'  Spoj {ci} + {cj}  (rastojanje: {dist:.3f})')

## 5. CAST (Cluster Affinity Search Technique)

Zasnovan na konceptu **pokvarenih klika** — u idealnom slučaju, geni u istom klasteru bi formirali klike u grafu sličnosti, ali greške u merenjima to narušavaju.

CAST **ne zahteva** unapred zadat broj klastera k. Koristi prag sličnosti **θ (theta)**.

**Algoritam:**
1. Konstruiši graf sličnosti G(R, θ)
2. Počni klaster sa čvorom maksimalnog stepena
3. Iterativno dodaj θ-bliske i ukloni θ-udaljene gene dok klaster nije konzistentan
4. Ukloni klaster i ponovi za preostale čvorove

Gen i je **θ-blizak** klasteru C ako je R(i,C) > θ, inače je **θ-udaljen**.

In [7]:
class CAST:
    def __init__(self, theta):
        self.theta = theta
        self.clusters = []
        self.similarity_matrix = None
    
    '''Izracunavanje matrice slicnosti (Pearsonova korelacija)'''
    def compute_similarity(self, data):
        n = len(data)
        R = [[0.0] * n for _ in range(n)]
        
        for i in range(n):
            R[i][i] = 1.0
            for j in range(i + 1, n):
                corr = pearson_correlation(data[i], data[j])
                R[i][j] = corr
                R[j][i] = corr
        
        self.similarity_matrix = R
        return R
    
    '''Prosecna slicnost gena prema klasteru: R(i,C)'''
    def similarity_with_cluster(self, gene, cluster, R):
        if not cluster:
            return 0.0
        return sum(R[gene][j] for j in cluster) / len(cluster)
    
    '''Stepen cvora u grafu slicnosti (broj suseda sa R > theta)'''
    def node_degree(self, node, remaining, R):
        return sum(1 for j in remaining if j != node and R[node][j] > self.theta)
    
    '''Pokretanje CAST algoritma'''
    def fit(self, data=None, similarity_matrix=None):
        if similarity_matrix is not None:
            R = similarity_matrix
            self.similarity_matrix = R
        elif data is not None:
            R = self.compute_similarity(data)
        else:
            raise ValueError("Mora biti dat ili podaci ili matrica_slicnosti")
        
        n = len(R)
        self.clusters = []
        remaining = set(range(n))
        
        while remaining:
            # Nadji cvor sa maksimalnim stepenom
            max_degree = -1
            starting = None
            for node in remaining:
                st = self.node_degree(node, remaining, R)
                if st > max_degree:
                    max_degree = st
                    starting = node
            
            cluster = {starting}
            
            # Iterativno dodaj teta-bliske i ukloni teta-udaljene
            changed = True
            while changed:
                changed = False
                
                # Dodaj najblizi teta-blizak gen koji nije u C
                best = None
                best_sim = -float('inf')
                for i in remaining:
                    if i in cluster:
                        continue
                    sl = self.similarity_with_cluster(i, cluster, R)
                    if sl > self.theta and sl > best_sim:
                        best_sim = sl
                        best = i
                
                if best is not None:
                    cluster.add(best)
                    changed = True
                
                # Ukloni najudaljeniji teta-udaljen gen iz C
                if len(cluster) > 1:
                    worst = None
                    worst_sim = float('inf')
                    for i in list(cluster):
                        others = cluster - {i}
                        sl = self.similarity_with_cluster(i, others, R)
                        if sl <= self.theta and sl < worst_sim:
                            worst_sim = sl
                            worst = i
                    
                    if worst is not None:
                        cluster.remove(worst)
                        changed = True
            
            self.clusters.append(sorted(cluster))
            remaining -= cluster
        
        return self.clusters
    
    '''Labele klastera za svaku tacku'''
    def labels(self, n=None):
        if n is None:
            n = max(max(c) for c in self.clusters) + 1
        
        lab = [-1] * n
        for cluster_id, kl in enumerate(self.clusters):
            for gene in kl:
                lab[gene] = cluster_id
        return lab

## Testovi

### Podaci genske ekspresije kvasca (diauksična promena)

Simulirani log2 fold change vektori ekspresije za 10 gena kvasca kroz 7 vremenskih tačaka tokom diauksične promene (DeRisi eksperiment, 1997).

Tri grupe gena:
- **Up-regulisani** (g0-g3): ekspresija raste — uključeni u respiraciju
- **Down-regulisani** (g4-g6): ekspresija opada — fermentacija se gasi  
- **Nepromenjeni** (g7-g9): housekeeping geni — ravna ekspresija

In [8]:
# Simulirani log2 fold change vektori ekspresije
# 7 vremenskih tacaka: -6h, -4h, -2h, 0h (diauksicna promena), +2h, +4h, +6h
genes = [
    # Up-regulisani geni (respiracija)
    [0.11, 0.43, 0.45, 1.89, 2.00, 3.32, 2.56],   # g0: YLR258W (glikogen sintaza)
    [0.20, 0.50, 0.60, 1.70, 2.10, 3.00, 2.80],   # g1
    [0.05, 0.30, 0.35, 1.50, 1.80, 2.90, 2.30],   # g2
    [0.15, 0.40, 0.50, 1.60, 1.90, 3.10, 2.60],   # g3
    # Down-regulisani geni (fermentacija)
    [0.09, -0.28, -0.15, -1.18, -1.59, -2.96, -3.08],  # g4: YPL012W
    [-0.10, -0.20, -0.30, -1.00, -1.40, -2.50, -2.80],  # g5
    [0.05, -0.15, -0.25, -1.30, -1.70, -3.10, -2.90],   # g6
    # Nepromenjeni geni (housekeeping)
    [0.15, 0.15, 0.17, 0.09, 0.07, 0.09, 0.07],   # g7: YPR055W
    [0.10, 0.08, 0.12, 0.05, 0.03, 0.06, 0.04],   # g8
    [-0.05, 0.02, -0.03, 0.08, -0.02, 0.01, 0.03], # g9
]

gene_names = ["YLR258W", "GEN_UP1", "GEN_UP2", "GEN_UP3",
              "YPL012W", "GEN_DN1", "GEN_DN2",
              "YPR055W", "GEN_FL1", "GEN_FL2"]

print(f"Broj gena: {len(genes)}")
print(f"Broj vremenskih tacaka: {len(genes[0])}")
print()
print("Matrica ekspresije (log2 fold change):")
print(f"{'Gen':<10} {'t1':>6} {'t2':>6} {'t3':>6} {'t4':>6} {'t5':>6} {'t6':>6} {'t7':>6}")
print("-" * 55)
for i, gene in enumerate(genes):
    values_str = " ".join(f"{v:>6.2f}" for v in gene)
    print(f"{gene_names[i]:<10} {values_str}")

Broj gena: 10
Broj vremenskih tacaka: 7

Matrica ekspresije (log2 fold change):
Gen            t1     t2     t3     t4     t5     t6     t7
-------------------------------------------------------
YLR258W      0.11   0.43   0.45   1.89   2.00   3.32   2.56
GEN_UP1      0.20   0.50   0.60   1.70   2.10   3.00   2.80
GEN_UP2      0.05   0.30   0.35   1.50   1.80   2.90   2.30
GEN_UP3      0.15   0.40   0.50   1.60   1.90   3.10   2.60
YPL012W      0.09  -0.28  -0.15  -1.18  -1.59  -2.96  -3.08
GEN_DN1     -0.10  -0.20  -0.30  -1.00  -1.40  -2.50  -2.80
GEN_DN2      0.05  -0.15  -0.25  -1.30  -1.70  -3.10  -2.90
YPR055W      0.15   0.15   0.17   0.09   0.07   0.09   0.07
GEN_FL1      0.10   0.08   0.12   0.05   0.03   0.06   0.04
GEN_FL2     -0.05   0.02  -0.03   0.08  -0.02   0.01   0.03


### Test 1: FarthestFirstTraversal

In [9]:
fft = FarthestFirstTraversal(k=3)
centers = fft.fit(genes)
labels = fft.assign_clusters(genes)

print("FarthestFirstTraversal (k=3)")
print("=" * 50)
for cluster_id in range(3):
    members = [gene_names[i] for i in range(len(genes)) if labels[i] == cluster_id]
    print(f"  Klaster {cluster_id + 1}: {members}")

FarthestFirstTraversal (k=3)
  Klaster 1: ['YLR258W', 'GEN_UP1', 'GEN_UP2', 'GEN_UP3']
  Klaster 2: ['YPL012W', 'GEN_DN1', 'GEN_DN2']
  Klaster 3: ['YPR055W', 'GEN_FL1', 'GEN_FL2']


### Test 2: Lloyd algoritam (K-means)

In [10]:
random.seed(42)
km = KMeans(k=3)
n_iter = km.fit(genes)

print("Lloyd algoritam / K-means (k=3)")
print("=" * 50)
print(f"Konvergirao nakon {n_iter} iteracija")
print(f"Distorzija: {km.distortion(genes):.4f}")
print()
for cluster_id in range(3):
    members = [gene_names[i] for i in range(len(genes)) if km.labels[i] == cluster_id]
    print(f"  Klaster {cluster_id + 1}: {members}")

print()
print("Centri klastera:")
for i, c in enumerate(km.centers):
    print(f"  Centar {i + 1}: [{', '.join(f'{x:.2f}' for x in c)}]")

Lloyd algoritam / K-means (k=3)
Konvergirao nakon 2 iteracija
Distorzija: 0.0852

  Klaster 1: ['YLR258W', 'GEN_UP1', 'GEN_UP2', 'GEN_UP3']
  Klaster 2: ['YPL012W', 'GEN_DN1', 'GEN_DN2']
  Klaster 3: ['YPR055W', 'GEN_FL1', 'GEN_FL2']

Centri klastera:
  Centar 1: [0.13, 0.41, 0.47, 1.67, 1.95, 3.08, 2.56]
  Centar 2: [0.01, -0.21, -0.23, -1.16, -1.56, -2.85, -2.93]
  Centar 3: [0.07, 0.08, 0.09, 0.07, 0.03, 0.05, 0.05]


### Test 3: Meki K-means (EM algoritam)

In [11]:
random.seed(42)
mkm = SoftKMeans(k=3, beta=2.0)
n_iter = mkm.fit(genes)

print("Meki K-means (k=3, beta=2.0)")
print("=" * 50)
print(f"Konvergirao nakon {n_iter} iteracija")
print()

# Matrica odgovornosti
print("Matrica odgovornosti (koliko svaki klaster 'privlaci' svaki gen):")
print(f"{'Gen':<10} {'Kl.1':>8} {'Kl.2':>8} {'Kl.3':>8}  → Kruti label")
print("-" * 55)

hard = mkm.hard_labels()
for j in range(len(genes)):
    resp = mkm.responsibilities(j)
    print(f"{gene_names[j]:<10} {resp[0]:>8.4f} {resp[1]:>8.4f} {resp[2]:>8.4f}  → Klaster {hard[j] + 1}")

print()
print("Uticaj parametra beta na krutost dodele:")
for beta in [0.5, 1.0, 2.0, 5.0, 10.0]:
    random.seed(42)
    mkm_t = SoftKMeans(k=3, beta=beta)
    mkm_t.fit(genes)
    
    # Prosecna maksimalna odgovornost
    mean_max = 0
    for j in range(len(genes)):
        resp = mkm_t.responsibilities(j)
        mean_max += max(resp)
    mean_max /= len(genes)
    
    print(f"  beta={beta:<5.1f} → prosecna maks. odgovornost: {mean_max:.4f} (1.0 = potpuno kruto)")

Meki K-means (k=3, beta=2.0)
Konvergirao nakon 3 iteracija

Matrica odgovornosti (koliko svaki klaster 'privlaci' svaki gen):
Gen            Kl.1     Kl.2     Kl.3  → Kruti label
-------------------------------------------------------
YLR258W      0.9999   0.0000   0.0001  → Klaster 1
GEN_UP1      0.9999   0.0000   0.0001  → Klaster 1
GEN_UP2      0.9996   0.0000   0.0004  → Klaster 1
GEN_UP3      0.9999   0.0000   0.0001  → Klaster 1
YPL012W      0.0000   0.9999   0.0001  → Klaster 2
GEN_DN1      0.0000   0.9995   0.0005  → Klaster 2
GEN_DN2      0.0000   0.9999   0.0001  → Klaster 2
YPR055W      0.0001   0.0001   0.9998  → Klaster 3
GEN_FL1      0.0001   0.0001   0.9998  → Klaster 3
GEN_FL2      0.0001   0.0002   0.9997  → Klaster 3

Uticaj parametra beta na krutost dodele:
  beta=0.5   → prosecna maks. odgovornost: 0.8348 (1.0 = potpuno kruto)
  beta=1.0   → prosecna maks. odgovornost: 0.9839 (1.0 = potpuno kruto)
  beta=2.0   → prosecna maks. odgovornost: 0.9998 (1.0 = potpuno krut

### Test 4: Hijerarhijsko klasterovanje

In [12]:
# Izracunaj matricu rastojanja (euklidsko)
n = len(genes)
D = [[0.0] * n for _ in range(n)]
for i in range(n):
    for j in range(i + 1, n):
        d = euclidean_distance(genes[i], genes[j])
        D[i][j] = d
        D[j][i] = d

# D_avg (UPGMA)
print("Hijerarhijsko klasterovanje (D_avg / UPGMA)")
print("=" * 50)
hk_avg = HierarchicalClustering(method='avg')
hk_avg.fit(D)
print("Redosled spajanja:")
hk_avg.print_dendrogram(gene_names)

print()
clusters_3 = hk_avg.extract_clusters(3)
print("Izdvojeni klasteri (k=3):")
for i, kl in enumerate(clusters_3):
    members = [gene_names[x] for x in kl]
    print(f"  Klaster {i + 1}: {members}")

# D_min
print()
print("Hijerarhijsko klasterovanje (D_min)")
print("=" * 50)
hk_min = HierarchicalClustering(method='min')
hk_min.fit(D)
print("Redosled spajanja:")
hk_min.print_dendrogram(gene_names)

print()
clusters_3_min = hk_min.extract_clusters(3)
print("Izdvojeni klasteri (k=3):")
for i, kl in enumerate(clusters_3_min):
    members = [gene_names[x] for x in kl]
    print(f"  Klaster {i + 1}: {members}")

Hijerarhijsko klasterovanje (D_avg / UPGMA)
Redosled spajanja:
  Spoj ['YPR055W'] + ['GEN_FL1']  (rastojanje: 0.122)
  Spoj ['GEN_FL2'] + ['YPR055W', 'GEN_FL1']  (rastojanje: 0.285)
  Spoj ['YPL012W'] + ['GEN_DN2']  (rastojanje: 0.327)
  Spoj ['GEN_UP1'] + ['GEN_UP3']  (rastojanje: 0.350)
  Spoj ['YLR258W'] + ['GEN_UP1', 'GEN_UP3']  (rastojanje: 0.439)
  Spoj ['GEN_UP2'] + ['YLR258W', 'GEN_UP1', 'GEN_UP3']  (rastojanje: 0.613)
  Spoj ['GEN_DN1'] + ['YPL012W', 'GEN_DN2']  (rastojanje: 0.705)
  Spoj ['GEN_FL2', 'YPR055W', 'GEN_FL1'] + ['GEN_DN1', 'YPL012W', 'GEN_DN2']  (rastojanje: 4.647)
  Spoj ['GEN_UP2', 'YLR258W', 'GEN_UP1', 'GEN_UP3'] + ['GEN_FL2', 'YPR055W', 'GEN_FL1', 'GEN_DN1', 'YPL012W', 'GEN_DN2']  (rastojanje: 7.005)

Izdvojeni klasteri (k=3):
  Klaster 1: ['GEN_UP2', 'YLR258W', 'GEN_UP1', 'GEN_UP3']
  Klaster 2: ['GEN_FL2', 'YPR055W', 'GEN_FL1']
  Klaster 3: ['GEN_DN1', 'YPL012W', 'GEN_DN2']

Hijerarhijsko klasterovanje (D_min)
Redosled spajanja:
  Spoj ['YPR055W'] + ['GEN_FL

### Test 5: CAST algoritam

In [13]:
cast = CAST(theta=0.7)
clusters = cast.fit(data=genes)

print("CAST algoritam (theta=0.7, Pearsonova korelacija)")
print("=" * 50)
print(f"Pronadeno {len(clusters)} klastera (automatski!)")
print()

for i, kl in enumerate(clusters):
    members = [gene_names[x] for x in kl]
    print(f"  Klaster {i + 1}: {members}")

# Izvod matrice slicnosti (Pearsonova korelacija)
print()
R = cast.similarity_matrix
print("Matrica Pearsonove korelacije (izvod):")
print(f"{'':>10}", end="")
for name in gene_names[:5]:
    print(f"{name:>10}", end="")
print(" ...")
for i in range(5):
    print(f"{gene_names[i]:<10}", end="")
    for j in range(5):
        print(f"{R[i][j]:>10.3f}", end="")
    print(" ...")

# Uticaj praga theta
print()
print("Uticaj praga theta:")
for theta in [0.5, 0.6, 0.7, 0.8, 0.9]:
    cast_t = CAST(theta=theta)
    clusters_t = cast_t.fit(data=genes)
    sizes = [len(c) for c in clusters_t]
    print(f"  theta={theta:.1f} → {len(clusters_t)} klastera, velicine: {sizes}")

CAST algoritam (theta=0.7, Pearsonova korelacija)
Pronadeno 3 klastera (automatski!)

  Klaster 1: ['YPL012W', 'GEN_DN1', 'GEN_DN2', 'YPR055W', 'GEN_FL1']
  Klaster 2: ['YLR258W', 'GEN_UP1', 'GEN_UP2', 'GEN_UP3']
  Klaster 3: ['GEN_FL2']

Matrica Pearsonove korelacije (izvod):
             YLR258W   GEN_UP1   GEN_UP2   GEN_UP3   YPL012W ...
YLR258W        1.000     0.988     0.998     0.995    -0.958 ...
GEN_UP1        0.988     1.000     0.994     0.996    -0.981 ...
GEN_UP2        0.998     0.994     1.000     0.999    -0.968 ...
GEN_UP3        0.995     0.996     0.999     1.000    -0.978 ...
YPL012W       -0.958    -0.981    -0.968    -0.978     1.000 ...

Uticaj praga theta:
  theta=0.5 → 3 klastera, velicine: [5, 4, 1]
  theta=0.6 → 3 klastera, velicine: [5, 4, 1]
  theta=0.7 → 3 klastera, velicine: [5, 4, 1]
  theta=0.8 → 4 klastera, velicine: [2, 4, 3, 1]
  theta=0.9 → 4 klastera, velicine: [4, 3, 2, 1]


### Test 6: Poređenje svih algoritama na istim podacima

In [14]:
print("POREDENJE SVIH ALGORITAMA")
print("=" * 70)
print(f"{'Algoritam':<30} {'Klaster 1':<20} {'Klaster 2':<20} {'Klaster 3':<20}")
print("-" * 70)

# Ocekivano: up=[0,1,2,3], down=[4,5,6], flat=[7,8,9]

def format_clusters(labels, names, k=3):
    result = []
    for cluster_id in range(k):
        members = [names[i] for i in range(len(names)) if labels[i] == cluster_id]
        result.append(members)
    # Sortiraj po prvom clanu da budu uporedivi
    result.sort(key=lambda x: x[0] if x else "")
    return result

# FFT
fft = FarthestFirstTraversal(k=3)
fft.fit(genes)
fft_labels = fft.assign_clusters(genes)
fft_kl = format_clusters(fft_labels, gene_names)

# K-means
random.seed(42)
km = KMeans(k=3)
km.fit(genes)
km_kl = format_clusters(km.labels, gene_names)

# Meki K-means
random.seed(42)
mkm = SoftKMeans(k=3, beta=2.0)
mkm.fit(genes)
mkm_kl = format_clusters(mkm.hard_labels(), gene_names)

# Hijerarhijsko (UPGMA)
D = [[0.0] * len(genes) for _ in range(len(genes))]
for i in range(len(genes)):
    for j in range(i + 1, len(genes)):
        d = euclidean_distance(genes[i], genes[j])
        D[i][j] = d
        D[j][i] = d
hk = HierarchicalClustering(method='avg')
hk.fit(D)
hk_clusters = hk.extract_clusters(3)
hk_labels = [0] * len(genes)
for cluster_id, kl in enumerate(hk_clusters):
    for idx in kl:
        hk_labels[idx] = cluster_id
hk_kl = format_clusters(hk_labels, gene_names)

# CAST
cast = CAST(theta=0.7)
cast.fit(data=genes)
cast_labels = cast.labels(len(genes))
cast_kl = format_clusters(cast_labels, gene_names, k=max(cast_labels)+1)

for label_name, clusters in [("FarthestFirstTraversal", fft_kl),
                         ("Lloyd (K-means)", km_kl),
                         ("Meki K-means (β=2)", mkm_kl),
                         ("Hijerarhijsko (UPGMA)", hk_kl),
                         ("CAST (θ=0.7)", cast_kl)]:
    # Skrati imena za prikaz
    cluster_strs = []
    for kl in clusters:
        shortened = [name[:6] for name in kl]
        cluster_strs.append(", ".join(shortened))
    
    # Dopuni do 3 kolone
    while len(cluster_strs) < 3:
        cluster_strs.append("-")
    
    print(f"{label_name:<30} {cluster_strs[0]:<20} {cluster_strs[1]:<20} {cluster_strs[2]:<20}")

print()
print("Ocekivano: Up=[YLR258, UP1-3], Down=[YPL012, DN1-2], Flat=[YPR055, FL1-2]")

POREDENJE SVIH ALGORITAMA
Algoritam                      Klaster 1            Klaster 2            Klaster 3           
----------------------------------------------------------------------
FarthestFirstTraversal         YLR258, GEN_UP, GEN_UP, GEN_UP YPL012, GEN_DN, GEN_DN YPR055, GEN_FL, GEN_FL
Lloyd (K-means)                YLR258, GEN_UP, GEN_UP, GEN_UP YPL012, GEN_DN, GEN_DN YPR055, GEN_FL, GEN_FL
Meki K-means (β=2)             YLR258, GEN_UP, GEN_UP, GEN_UP YPL012, GEN_DN, GEN_DN YPR055, GEN_FL, GEN_FL
Hijerarhijsko (UPGMA)          YLR258, GEN_UP, GEN_UP, GEN_UP YPL012, GEN_DN, GEN_DN YPR055, GEN_FL, GEN_FL
CAST (θ=0.7)                   GEN_FL               YLR258, GEN_UP, GEN_UP, GEN_UP YPL012, GEN_DN, GEN_DN, YPR055, GEN_FL

Ocekivano: Up=[YLR258, UP1-3], Down=[YPL012, DN1-2], Flat=[YPR055, FL1-2]


### Test 7: Jednostavan primer u 2D prostoru (iz udžbenika)

Osam tačaka u dvodimenzionalnom prostoru sa Slike 8.6 iz udžbenika.

In [15]:
# Tacke iz Slike 8.6 (desno) — osam tacaka u 2D
points_2d = [
    [1, 6], [1, 3], [3, 4], [5, 6], [5, 2], [7, 1], [8, 7], [10, 3]
]

print("Osam tacaka u 2D prostoru:")
for i, t in enumerate(points_2d):
    print(f"  Tacka {i}: ({t[0]}, {t[1]})")

print()

# FarthestFirstTraversal
fft2 = FarthestFirstTraversal(k=3)
fft2.fit(points_2d)
labels2 = fft2.assign_clusters(points_2d)
print("FarthestFirstTraversal (k=3):")
for kl in range(3):
    cl = [points_2d[i] for i in range(len(points_2d)) if labels2[i] == kl]
    print(f"  Klaster {kl+1}: {cl}")

print()

# K-means
random.seed(0)
km2 = KMeans(k=3)
km2.fit(points_2d)
print("Lloyd/K-means (k=3):")
for kl in range(3):
    cl = [points_2d[i] for i in range(len(points_2d)) if km2.labels[i] == kl]
    print(f"  Klaster {kl+1}: {cl}")
print(f"  Distorzija: {km2.distortion(points_2d):.4f}")

print()

# Hijerarhijsko
n2 = len(points_2d)
D2 = [[0.0]*n2 for _ in range(n2)]
for i in range(n2):
    for j in range(i+1, n2):
        d = euclidean_distance(points_2d[i], points_2d[j])
        D2[i][j] = d
        D2[j][i] = d

hk2 = HierarchicalClustering(method='avg')
hk2.fit(D2)
print("Hijerarhijsko (UPGMA) → redosled spajanja:")
hk2.print_dendrogram()
print()
clusters2 = hk2.extract_clusters(3)
print(f"  3 klastera: {clusters2}")

Osam tacaka u 2D prostoru:
  Tacka 0: (1, 6)
  Tacka 1: (1, 3)
  Tacka 2: (3, 4)
  Tacka 3: (5, 6)
  Tacka 4: (5, 2)
  Tacka 5: (7, 1)
  Tacka 6: (8, 7)
  Tacka 7: (10, 3)

FarthestFirstTraversal (k=3):
  Klaster 1: [[1, 6], [1, 3], [3, 4], [5, 6]]
  Klaster 2: [[8, 7], [10, 3]]
  Klaster 3: [[5, 2], [7, 1]]

Lloyd/K-means (k=3):
  Klaster 1: [[5, 6], [8, 7], [10, 3]]
  Klaster 2: [[5, 2], [7, 1]]
  Klaster 3: [[1, 6], [1, 3], [3, 4]]
  Distorzija: 3.8958

Hijerarhijsko (UPGMA) → redosled spajanja:
  Spoj [1] + [2]  (rastojanje: 2.236)
  Spoj [4] + [5]  (rastojanje: 2.236)
  Spoj [0] + [1, 2]  (rastojanje: 2.914)
  Spoj [3] + [6]  (rastojanje: 3.162)
  Spoj [7] + [4, 5]  (rastojanje: 4.352)
  Spoj [3, 6] + [7, 4, 5]  (rastojanje: 5.267)
  Spoj [0, 1, 2] + [3, 6, 7, 4, 5]  (rastojanje: 6.006)

  3 klastera: [[0, 1, 2], [3, 6], [7, 4, 5]]


### Test 8: Partition funkcija — jednodimenzionalni primer (Slika 8.21)

Pet tačaka Data = (-3, -2, 0, +2, +3) sa dva centra na x₁ = -2.5 i x₂ = +2.5.

Demonstrira kako parametar krutosti β kontroliše mekost dodele.

In [16]:
# Jednodimenzionalni primer iz Slike 8.21
data_1d = [[-3], [-2], [0], [2], [3]]
centers_1d = [[-2.5], [2.5]]

print("Partition funkcija — Slika 8.21")
print("Data = (-3, -2, 0, +2, +3),  Centers = {-2.5, +2.5}")
print("=" * 60)

for beta in [0.5, 1.0, 2.0, 5.0]:
    print(f"\nbeta = {beta}")
    print(f"  {'Tacka':>6}  {'Odg(x1=-2.5)':>14}  {'Odg(x2=+2.5)':>14}")
    print("  " + "-" * 38)
    
    for j, point in enumerate(data_1d):
        # Partition funkcija: e^(-beta*d) / sum(e^(-beta*d))
        exponents = []
        for center in centers_1d:
            d = abs(point[0] - center[0])
            exponents.append(-beta * d)
        
        max_val = max(exponents)
        exp_vals = [math.exp(e - max_val) for e in exponents]
        sum_total = sum(exp_vals)
        
        resp = [e / sum_total for e in exp_vals]
        print(f"  {point[0]:>6}  {resp[0]:>14.4f}  {resp[1]:>14.4f}")

Partition funkcija — Slika 8.21
Data = (-3, -2, 0, +2, +3),  Centers = {-2.5, +2.5}

beta = 0.5
   Tacka    Odg(x1=-2.5)    Odg(x2=+2.5)
  --------------------------------------
      -3          0.9241          0.0759
      -2          0.8808          0.1192
       0          0.5000          0.5000
       2          0.1192          0.8808
       3          0.0759          0.9241

beta = 1.0
   Tacka    Odg(x1=-2.5)    Odg(x2=+2.5)
  --------------------------------------
      -3          0.9933          0.0067
      -2          0.9820          0.0180
       0          0.5000          0.5000
       2          0.0180          0.9820
       3          0.0067          0.9933

beta = 2.0
   Tacka    Odg(x1=-2.5)    Odg(x2=+2.5)
  --------------------------------------
      -3          1.0000          0.0000
      -2          0.9997          0.0003
       0          0.5000          0.5000
       2          0.0003          0.9997
       3          0.0000          1.0000

beta = 5.0
   Tack

## 6. DBSCAN (Density-Based Spatial Clustering)

Density-based algoritam koji pronalazi klastere **proizvoljnog oblika** i identifikuje **šum** (outlier-e). Za razliku od k-means i hijerarhijskog klasterovanja, broj klastera nije zadat — proizilazi iz strukture podataka.

**Algoritam:**
1. Za svaku neoznačenu tačku P pronađi ε-okolinu N(P)
2. Ako |N(P)| < MinPts → P je **šum** (label = -1)
3. Inače, P je **jezgreni** čvor → kreiraj novi klaster C i proširi ga BFS-om kroz gusto-povezane susede
4. Šum može kasnije postati **granična** tačka ako se priključi nekom klasteru

**Bioinformatički kontekst:** korisno za pronalaženje gustih familija proteina ili gena kada ne znamo unapred koliko ima funkcionalnih grupa, i kada ima atipičnih sekvenci.

In [17]:
'''Pomocna oznaka stanja tacke u DBSCAN-u: 0 = nije razmatrana, -1 = sum, >0 = ID klastera'''
class DBSCAN:
    def __init__(self, eps, min_pts):
        self.eps = eps
        self.min_pts = min_pts
        self.labels = []

    '''Pronalazenje suseda tacke u epsilon-okolini'''
    def eps_neighbors(self, idx_val, data):
        neighbors = []
        for j in range(len(data)):
            if euclidean_distance(data[idx_val], data[j]) <= self.eps:
                neighbors.append(j)
        return neighbors

    '''Sirenje klastera C iz seme P kroz gusto-povezane tacke (BFS)'''
    def expand_cluster(self, data, p, neighbors, c):
        self.labels[p] = c

        i = 0
        while i < len(neighbors):
            q = neighbors[i]

            if self.labels[q] == -1:
                # Bivsi sum postaje granicna tacka klastera
                self.labels[q] = c
            elif self.labels[q] == 0:
                self.labels[q] = c
                new_neighbors = self.eps_neighbors(q, data)
                if len(new_neighbors) >= self.min_pts:
                    # q je novi jezgreni cvor — prosiri front
                    neighbors = neighbors + new_neighbors

            i += 1

    '''Pokretanje DBSCAN algoritma'''
    def fit(self, data):
        n = len(data)
        self.labels = [0] * n
        c = 0

        for p in range(n):
            if self.labels[p] != 0:
                continue

            neighbors = self.eps_neighbors(p, data)

            if len(neighbors) < self.min_pts:
                self.labels[p] = -1
            else:
                c += 1
                self.expand_cluster(data, p, neighbors, c)

        return self.labels

    '''Broj pronadjenih klastera (bez suma)'''
    def num_clusters(self):
        return max(self.labels) if self.labels else 0

    '''Indeksi tacaka oznacenih kao sum'''
    def noise_points(self):
        return [i for i, l in enumerate(self.labels) if l == -1]

### Test 9: DBSCAN

Različite vrednosti `eps` i `min_pts` pokazuju kako se klasteri formiraju i koje tačke su šum.

In [18]:
print("DBSCAN algoritam — uticaj parametara")
print("=" * 50)

for eps, min_pts in [(1.0, 2), (1.5, 2), (2.0, 2), (1.5, 3)]:
    db = DBSCAN(eps=eps, min_pts=min_pts)
    labels = db.fit(genes)

    n_clusters = db.num_clusters()
    noise_idx = db.noise_points()

    print(f"\neps={eps}, min_pts={min_pts}  →  {n_clusters} klastera, {len(noise_idx)} sum")

    for cluster_id in range(1, n_clusters + 1):
        members = [gene_names[i] for i in range(len(genes)) if labels[i] == cluster_id]
        print(f"  Klaster {cluster_id}: {members}")

    if noise_idx:
        noise_genes = [gene_names[i] for i in noise_idx]
        print(f"  Sum: {noise_genes}")

DBSCAN algoritam — uticaj parametara

eps=1.0, min_pts=2  →  3 klastera, 0 sum
  Klaster 1: ['YLR258W', 'GEN_UP1', 'GEN_UP2', 'GEN_UP3']
  Klaster 2: ['YPL012W', 'GEN_DN1', 'GEN_DN2']
  Klaster 3: ['YPR055W', 'GEN_FL1', 'GEN_FL2']

eps=1.5, min_pts=2  →  3 klastera, 0 sum
  Klaster 1: ['YLR258W', 'GEN_UP1', 'GEN_UP2', 'GEN_UP3']
  Klaster 2: ['YPL012W', 'GEN_DN1', 'GEN_DN2']
  Klaster 3: ['YPR055W', 'GEN_FL1', 'GEN_FL2']

eps=2.0, min_pts=2  →  3 klastera, 0 sum
  Klaster 1: ['YLR258W', 'GEN_UP1', 'GEN_UP2', 'GEN_UP3']
  Klaster 2: ['YPL012W', 'GEN_DN1', 'GEN_DN2']
  Klaster 3: ['YPR055W', 'GEN_FL1', 'GEN_FL2']

eps=1.5, min_pts=3  →  3 klastera, 0 sum
  Klaster 1: ['YLR258W', 'GEN_UP1', 'GEN_UP2', 'GEN_UP3']
  Klaster 2: ['YPL012W', 'GEN_DN1', 'GEN_DN2']
  Klaster 3: ['YPR055W', 'GEN_FL1', 'GEN_FL2']


## 7. DIANA (Divisive Analysis — top-down hijerarhijsko)

Komplementarni pristup aglomerativnom hijerarhijskom klasterovanju: umesto spajanja od dna nagore, DIANA **deli** klastere od korena nadole.

**Algoritam (klasična DIANA):**
1. Krenuti sa jednim klasterom koji sadrži sve tačke
2. Pronaći klaster sa najvećim **prečnikom** (max rastojanje između bilo koje dve tačke u klasteru)
3. U njemu pronaći tačku sa najvećim prosečnim rastojanjem do ostalih → **osnivač splinter grupe**
4. Iterativno premeštati tačke iz ostatka u splinter ako im je prosečno rastojanje do splintera manje nego do ostatka
5. Ponavljati 2–4 dok ne dobijemo k klastera (ili same singletone)

In [19]:
'''Pomocna klasa za DIANA klaster: visina = precnik u trenutku podele'''
class DCluster:
    def __init__(self, elements, height=0):
        self.elements = elements
        self.height = height
        self.left = None
        self.right = None

    def __str__(self):
        return f'{self.elements}'

    '''Precnik klastera: maksimalno rastojanje izmedu dva njegova clana'''
    @staticmethod
    def diameter(cluster, D):
        max_val = 0.0
        elements = cluster.elements
        for a in range(len(elements)):
            for b in range(a + 1, len(elements)):
                if D[elements[a]][elements[b]] > max_val:
                    max_val = D[elements[a]][elements[b]]
        return max_val


class DIANA:
    def __init__(self):
        self.root = None
        self.history = []

    '''Prosecno rastojanje tacke i do svih tacaka iz skupa S (iskljucujuci samu sebe)'''
    def mean_to_set(self, i, S, D):
        others = [j for j in S if j != i]
        if not others:
            return 0.0
        return sum(D[i][j] for j in others) / len(others)

    '''Podela klastera na splinter grupu i ostatak (Kaufman-Rousseeuw)'''
    def split_cluster(self, cluster, D):
        elements = cluster.elements[:]

        # Osnivac splinter grupe: tacka sa najvecim prosecnim rastojanjem
        max_mean = -1
        founder = elements[0]
        for i in elements:
            p = self.mean_to_set(i, elements, D)
            if p > max_mean:
                max_mean = p
                founder = i

        splinter = [founder]
        rest = [i for i in elements if i != founder]

        # Iterativno premestaj tacke koje su blize splinteru nego ostatku
        changed = True
        while changed and rest:
            changed = False
            best = None
            largest_diff = 0

            for i in rest:
                a = self.mean_to_set(i, rest, D)
                b = self.mean_to_set(i, splinter, D)
                diff = a - b
                if diff > largest_diff:
                    largest_diff = diff
                    best = i

            if best is not None:
                splinter.append(best)
                rest.remove(best)
                changed = True

        return splinter, rest

    '''Pronalazenje klastera sa najvecim precnikom (kandidat za podelu)'''
    def most_diverse_cluster(self, clusters, D):
        candidates = [c for c in clusters if len(c.elements) > 1]
        if not candidates:
            return None, 0.0

        max_d = -1
        chosen = candidates[0]
        for c in candidates:
            p = DCluster.diameter(c, D)
            if p > max_d:
                max_d = p
                chosen = c

        return chosen, max_d

    '''Pokretanje DIANA algoritma — deljenje dok svi klasteri ne postanu singltoni'''
    def fit(self, D):
        n = len(D)
        self.root = DCluster(list(range(n)))
        self.history = []

        clusters = [self.root]

        while True:
            chosen, diameter = self.most_diverse_cluster(clusters, D)
            if chosen is None:
                break

            splinter, rest = self.split_cluster(chosen, D)

            chosen.height = diameter
            chosen.left = DCluster(splinter)
            chosen.right = DCluster(rest)

            self.history.append((splinter, rest, diameter))

            clusters.remove(chosen)
            clusters.append(chosen.left)
            clusters.append(chosen.right)

        return self.root

    '''Izdvajanje k klastera iz dendrograma — secenje na najvisim podelama'''
    def extract_clusters(self, k):
        nodes = [self.root]

        while len(nodes) < k:
            max_h = -1
            max_idx = -1
            for idx, c in enumerate(nodes):
                if c.left is not None and c.height > max_h:
                    max_h = c.height
                    max_idx = idx

            if max_idx < 0:
                break

            node = nodes.pop(max_idx)
            nodes.append(node.left)
            nodes.append(node.right)

        return [c.elements for c in nodes]

    '''Ispis dendrograma (redosled podela, od najvece)'''
    def print_dendrogram(self, names=None):
        for splinter, rest, p in self.history:
            if names:
                splinter_name = [names[x] for x in splinter]
                rest_name = [names[x] for x in rest]
                print(f'  Podela: {splinter_name} | {rest_name}  (precnik: {p:.3f})')
            else:
                print(f'  Podela: {splinter} | {rest}  (precnik: {p:.3f})')

### Test 10: DIANA — podela od korena nadole

In [20]:
# Iskoristi prethodno izracunatu matricu rastojanja D
diana = DIANA()
diana.fit(D)

print("DIANA — divisive hijerarhijsko klasterovanje")
print("=" * 50)
print("Redosled podela:")
diana.print_dendrogram(gene_names)

print()
clusters_3 = diana.extract_clusters(3)
print("Izdvojeni klasteri (k=3):")
for i, kl in enumerate(clusters_3):
    members = [gene_names[x] for x in kl]
    print(f"  Klaster {i + 1}: {members}")

DIANA — divisive hijerarhijsko klasterovanje
Redosled podela:
  Podela: ['GEN_DN2', 'YPL012W', 'GEN_DN1'] | ['YLR258W', 'GEN_UP1', 'GEN_UP2', 'GEN_UP3', 'YPR055W', 'GEN_FL1', 'GEN_FL2']  (precnik: 9.784)
  Podela: ['GEN_FL2', 'GEN_FL1', 'YPR055W'] | ['YLR258W', 'GEN_UP1', 'GEN_UP2', 'GEN_UP3']  (precnik: 5.014)
  Podela: ['GEN_DN1'] | ['GEN_DN2', 'YPL012W']  (precnik: 0.760)
  Podela: ['GEN_UP2'] | ['YLR258W', 'GEN_UP1', 'GEN_UP3']  (precnik: 0.718)
  Podela: ['YLR258W'] | ['GEN_UP1', 'GEN_UP3']  (precnik: 0.492)
  Podela: ['GEN_UP1'] | ['GEN_UP3']  (precnik: 0.350)
  Podela: ['GEN_FL2'] | ['GEN_FL1', 'YPR055W']  (precnik: 0.336)
  Podela: ['GEN_DN2'] | ['YPL012W']  (precnik: 0.327)
  Podela: ['GEN_FL1'] | ['YPR055W']  (precnik: 0.122)

Izdvojeni klasteri (k=3):
  Klaster 1: ['GEN_DN2', 'YPL012W', 'GEN_DN1']
  Klaster 2: ['GEN_FL2', 'GEN_FL1', 'YPR055W']
  Klaster 3: ['YLR258W', 'GEN_UP1', 'GEN_UP2', 'GEN_UP3']


## 8. Louvain i Leiden (community detection na grafu sličnosti)

Algoritmi za detekciju zajednica u **mrežama** — geni se modeluju kao čvorovi, a težinske grane se povlače između sličnih gena (npr. Pearsonova korelacija > prag). Cilj je particija koja maksimizuje **modularnost** Q:

$$Q = \sum_{c} \left[ \frac{\Sigma_{in}^{c}}{2m} - \left( \frac{\Sigma_{tot}^{c}}{2m} \right)^2 \right]$$

**Louvain** — pohlepna lokalna optimizacija: za svaki čvor proba ga premestiti u zajednicu suseda, prihvata premeštanje ako povećava Q. Ponavlja dok nema poboljšanja.

**Leiden** — popravlja Louvain dodajući **fazu prečišćavanja** unutar zajednica (sprečava odvajanje povezanih komponenti).

In [21]:
'''Pomocna klasa za reprezentaciju tezinskog grafa slicnosti'''
class SimilarityGraph:
    def __init__(self):
        self.neighbors = {}    # cvor -> {sused: tezina}

    '''Dodavanje cvora bez suseda'''
    def add_node(self, u):
        if u not in self.neighbors:
            self.neighbors[u] = {}

    '''Dodavanje tezinske grane (neusmerene)'''
    def add_edge(self, u, v, w):
        self.add_node(u)
        self.add_node(v)
        if u != v:
            self.neighbors[u][v] = w
            self.neighbors[v][u] = w

    '''Zbir tezina svih incidentnih ivica cvora (tezinski stepen)'''
    def degree(self, u):
        if u not in self.neighbors:
            return 0.0
        return sum(self.neighbors[u].values())

    '''2m: zbir tezina svih grana grafa (svaka brojana dvaput)'''
    def total_weight_2m(self):
        m2 = 0.0
        for u in self.neighbors:
            for w in self.neighbors[u].values():
                m2 += w
        return m2

    '''Konstrukcija grafa iz matrice slicnosti — grane samo gde R > prag'''
    @staticmethod
    def from_similarity_matrix(R, threshold=0.0):
        G = SimilarityGraph()
        n = len(R)
        for u in range(n):
            G.add_node(u)
        for u in range(n):
            for v in range(u + 1, n):
                if R[u][v] > threshold:
                    G.add_edge(u, v, R[u][v])
        return G


class Louvain:
    def __init__(self, similarity_threshold=0.0):
        self.similarity_threshold = similarity_threshold
        self.communities = {}
        self.modularity_value = 0.0

    '''Modularnost particije P (cvor -> zajednica)'''
    def modularity(self, G, P):
        m2 = G.total_weight_2m()
        if m2 == 0:
            return 0.0

        comm_inside = {}    # 2x tezina ivica unutar zajednice
        comm_degree = {}    # ukupni tezinski stepen zajednice

        for u in G.neighbors:
            comm = P[u]
            comm_degree[comm] = comm_degree.get(comm, 0.0) + G.degree(u)
            for v, w in G.neighbors[u].items():
                if P[v] == comm:
                    comm_inside[comm] = comm_inside.get(comm, 0.0) + w

        Q = 0.0
        for comm in set(P.values()):
            inside = comm_inside.get(comm, 0.0)
            total = comm_degree.get(comm, 0.0)
            Q += (inside / m2) - (total / m2) ** 2
        return Q

    '''Jedan prolaz lokalne optimizacije: pomeraj svaki cvor u susednu zajednicu sa najvecim deltaQ'''
    def local_optimization(self, G, P):
        changed_globally = False
        changed = True

        while changed:
            changed = False
            for u in list(G.neighbors.keys()):
                current = P[u]
                neighbor_comms = set(P[v] for v in G.neighbors[u])
                neighbor_comms.add(current)

                Q_current = self.modularity(G, P)
                best = current
                best_delta = 0.0

                for comm in neighbor_comms:
                    if comm == current:
                        continue
                    P[u] = comm
                    delta = self.modularity(G, P) - Q_current
                    if delta > best_delta:
                        best_delta = delta
                        best = comm

                P[u] = best
                if best != current:
                    changed = True
                    changed_globally = True

        return changed_globally

    '''Pre-numerisanje zajednica u uzastopne brojeve (0, 1, 2, ...)'''
    def normalize_labels(self, P):
        mapping = {}
        for u in P:
            if P[u] not in mapping:
                mapping[P[u]] = len(mapping)
        return {u: mapping[P[u]] for u in P}

    '''Pokretanje Louvain algoritma na podacima (graf iz Pearsonove korelacije)'''
    def fit(self, data):
        n = len(data)
        R = [[0.0] * n for _ in range(n)]
        for i in range(n):
            R[i][i] = 1.0
            for j in range(i + 1, n):
                r = pearson_correlation(data[i], data[j])
                R[i][j] = r
                R[j][i] = r

        G = SimilarityGraph.from_similarity_matrix(R, self.similarity_threshold)

        # Inicijalna particija: svaki cvor je svoja zajednica
        P = {u: u for u in G.neighbors}

        # Pohlepna lokalna optimizacija
        self.local_optimization(G, P)

        self.communities = self.normalize_labels(P)
        self.modularity_value = self.modularity(G, self.communities)
        return self.communities

    '''Labele zajednica kao lista po indeksu tacke'''
    def labels(self, n):
        return [self.communities.get(i, -1) for i in range(n)]


class Leiden(Louvain):
    '''Faza preciscavanja: unutar svake zajednice ponovo pokreni lokalnu optimizaciju (sa singltonima)'''
    def refine(self, G, P):
        # Grupisi cvorove po trenutnoj zajednici
        comm_members = {}
        for u, comm in P.items():
            comm_members.setdefault(comm, []).append(u)

        new_P = {}
        next_id = 0

        for comm, members in comm_members.items():
            # Indukovani podgraf samo nad clanovima ove zajednice
            subgraph = SimilarityGraph()
            for u in members:
                subgraph.add_node(u)
            for u in members:
                for v, w in G.neighbors[u].items():
                    if v in members and v > u:
                        subgraph.add_edge(u, v, w)

            # Inicijalno svaki cvor podgrafa je svoja podzajednica
            sub_P = {u: u for u in members}

            # Pokusaj lokalnu optimizaciju unutar zajednice
            self.local_optimization(subgraph, sub_P)

            # Mapiraj novo-detektovane podzajednice u globalne ID-jeve
            local_ids = {}
            for u in members:
                sub_comm = sub_P[u]
                if sub_comm not in local_ids:
                    local_ids[sub_comm] = next_id
                    next_id += 1
                new_P[u] = local_ids[sub_comm]

        return new_P

    '''Pokretanje Leiden algoritma: Louvain + faza preciscavanja'''
    def fit(self, data):
        n = len(data)
        R = [[0.0] * n for _ in range(n)]
        for i in range(n):
            R[i][i] = 1.0
            for j in range(i + 1, n):
                r = pearson_correlation(data[i], data[j])
                R[i][j] = r
                R[j][i] = r

        G = SimilarityGraph.from_similarity_matrix(R, self.similarity_threshold)

        P = {u: u for u in G.neighbors}
        self.local_optimization(G, P)

        # Faza preciscavanja — sprecava odvajanje povezanih komponenti
        P = self.refine(G, P)

        # Jos jedan prolaz lokalne optimizacije nad preciscenom particijom
        self.local_optimization(G, P)

        self.communities = self.normalize_labels(P)
        self.modularity_value = self.modularity(G, self.communities)
        return self.communities

### Test 11: Louvain i Leiden — community detection na grafu sličnosti gena

In [22]:
print("Louvain i Leiden — detekcija zajednica")
print("=" * 50)

for threshold in [0.5, 0.7]:
    print(f"\nPrag slicnosti (Pearson): {threshold}")
    print("-" * 40)

    # Louvain
    lv = Louvain(similarity_threshold=threshold)
    P_lv = lv.fit(genes)
    print(f"Louvain → Q = {lv.modularity_value:.4f}")

    comm_groups = {}
    for i, z in enumerate(lv.labels(len(genes))):
        comm_groups.setdefault(z, []).append(gene_names[i])
    for comm_id, members in sorted(comm_groups.items()):
        print(f"  Zajednica {comm_id}: {members}")

    # Leiden
    ld = Leiden(similarity_threshold=threshold)
    P_ld = ld.fit(genes)
    print(f"\nLeiden → Q = {ld.modularity_value:.4f}")

    comm_groups = {}
    for i, z in enumerate(ld.labels(len(genes))):
        comm_groups.setdefault(z, []).append(gene_names[i])
    for comm_id, members in sorted(comm_groups.items()):
        print(f"  Zajednica {comm_id}: {members}")

Louvain i Leiden — detekcija zajednica

Prag slicnosti (Pearson): 0.5
----------------------------------------
Louvain → Q = 0.4837
  Zajednica 0: ['YLR258W', 'GEN_UP1', 'GEN_UP2', 'GEN_UP3']
  Zajednica 1: ['YPL012W', 'GEN_DN1', 'GEN_DN2', 'YPR055W', 'GEN_FL1']
  Zajednica 2: ['GEN_FL2']

Leiden → Q = 0.4837
  Zajednica 0: ['YLR258W', 'GEN_UP1', 'GEN_UP2', 'GEN_UP3']
  Zajednica 1: ['YPL012W', 'GEN_DN1', 'GEN_DN2', 'YPR055W', 'GEN_FL1']
  Zajednica 2: ['GEN_FL2']

Prag slicnosti (Pearson): 0.7
----------------------------------------
Louvain → Q = 0.4837
  Zajednica 0: ['YLR258W', 'GEN_UP1', 'GEN_UP2', 'GEN_UP3']
  Zajednica 1: ['YPL012W', 'GEN_DN1', 'GEN_DN2', 'YPR055W', 'GEN_FL1']
  Zajednica 2: ['GEN_FL2']

Leiden → Q = 0.4837
  Zajednica 0: ['YLR258W', 'GEN_UP1', 'GEN_UP2', 'GEN_UP3']
  Zajednica 1: ['YPL012W', 'GEN_DN1', 'GEN_DN2', 'YPR055W', 'GEN_FL1']
  Zajednica 2: ['GEN_FL2']


## 9. Brute-force K-means (iscrpno traženje optimuma)

Lloyd algoritam je heuristika — može da zaglavi u lokalnom minimumu. **Brute-force** verzija pokušava da garantuje optimum. Naša implementacija ispituje **sve podskupove od k tačaka iz podataka kao kandidate za centre** (C(n, k) kombinacija), i bira onaj sa minimalnom distorzijom.

**Algoritam:**
1. Za svaki podskup od `k` tačaka u podacima:
2. Tretiraj te tačke kao centre, dodeli ostale po najbližem, izračunaj distorziju
3. Vrati izbor sa minimalnom distorzijom

**Složenost:** O(C(n,k) · n · k · d). Praktično samo za male n.

**Napomena:** Lloyd algoritam koristi **centroide** (sredine klastera), koji ne moraju biti tačke iz podataka — pa Lloyd može dati i manju distorziju od ove brute-force varijante. Ono što ovaj algoritam garantuje je optimum **u ograničenju da centri moraju biti iz skupa podataka** (k-medoids varijanta).

In [23]:
class BruteForceKMeans:
    def __init__(self, k):
        self.k = k
        self.centers = []
        self.labels = []
        self.best_distortion = float('inf')

    '''Generator svih C(n, k) kombinacija indeksa (rekurzivno, bez itertools)'''
    def combinations(self, n, k, start=0):
        if k == 0:
            yield []
            return
        for i in range(start, n - k + 1):
            for rest in self.combinations(n, k - 1, i + 1):
                yield [i] + rest

    '''Distorzija: prosecno kvadratno rastojanje tacke do najblizeg centra'''
    def distortion(self, data, centers):
        total = 0.0
        for point in data:
            min_r = min(euclidean_distance(point, c) for c in centers)
            total += min_r ** 2
        return total / len(data)

    '''Dodela tacaka klasterima po najblizem centru'''
    def assign_clusters(self, data):
        self.labels = []
        for point in data:
            min_r = float('inf')
            best_idx = 0
            for i, c in enumerate(self.centers):
                r = euclidean_distance(point, c)
                if r < min_r:
                    min_r = r
                    best_idx = i
            self.labels.append(best_idx)

    '''Pokretanje iscrpne pretrage: isprobaj sve C(n, k) podskupove tacaka kao centre'''
    def fit(self, data):
        n = len(data)
        self.best_distortion = float('inf')
        self.centers = []

        n_tried = 0
        for indices in self.combinations(n, self.k):
            candidate_centers = [data[i][:] for i in indices]
            d = self.distortion(data, candidate_centers)
            n_tried += 1

            if d < self.best_distortion:
                self.best_distortion = d
                self.centers = candidate_centers

        self.assign_clusters(data)
        return n_tried

### Test 12: Brute-force K-means vs Lloyd

Brute-force ovde traži optimalne **medoide** (centri = tačke iz podataka), dok Lloyd računa **centroide** (sredine). Zato Lloyd može da postigne i manju distorziju.

In [24]:
bfkm = BruteForceKMeans(k=3)
n_tried = bfkm.fit(genes)

print("Brute-force K-means (k=3)")
print("=" * 50)
print(f"Isprobano kombinacija: {n_tried} (C({len(genes)},3))")
print(f"Najmanja distorzija: {bfkm.best_distortion:.4f}")
print()
for cluster_id in range(3):
    members = [gene_names[i] for i in range(len(genes)) if bfkm.labels[i] == cluster_id]
    print(f"  Klaster {cluster_id + 1}: {members}")

print()
print("Najbolji centri (tacke iz podataka):")
for i, c in enumerate(bfkm.centers):
    print(f"  Centar {i + 1}: [{', '.join(f'{x:.2f}' for x in c)}]")

# Poredjenje sa Lloyd-om
print()
print("Poredjenje sa Lloyd algoritmom:")
random.seed(42)
km_cmp = KMeans(k=3)
km_cmp.fit(genes)
print(f"  Lloyd distorzija:        {km_cmp.distortion(genes):.4f}")
print(f"  Brute-force distorzija:  {bfkm.best_distortion:.4f}")
diff = km_cmp.distortion(genes) - bfkm.best_distortion
print(f"  Razlika:                 {diff:+.4f}")
if diff < 0:
    print("  (Lloyd je nizi jer koristi centroide, ne tacke iz podataka.)")
elif diff > 0:
    print("  (Brute-force je nizi — Lloyd je zaglavio u lokalnom minimumu.)")

Brute-force K-means (k=3)
Isprobano kombinacija: 120 (C(10,3))
Najmanja distorzija: 0.1064

  Klaster 1: ['YLR258W', 'GEN_UP1', 'GEN_UP2', 'GEN_UP3']
  Klaster 2: ['YPL012W', 'GEN_DN1', 'GEN_DN2']
  Klaster 3: ['YPR055W', 'GEN_FL1', 'GEN_FL2']

Najbolji centri (tacke iz podataka):
  Centar 1: [0.15, 0.40, 0.50, 1.60, 1.90, 3.10, 2.60]
  Centar 2: [0.09, -0.28, -0.15, -1.18, -1.59, -2.96, -3.08]
  Centar 3: [0.10, 0.08, 0.12, 0.05, 0.03, 0.06, 0.04]

Poredjenje sa Lloyd algoritmom:
  Lloyd distorzija:        0.0852
  Brute-force distorzija:  0.1064
  Razlika:                 -0.0212
  (Lloyd je nizi jer koristi centroide, ne tacke iz podataka.)
